In [ ]:
#!pip install datasets evaluate
#!pip install scikit-learn
#!pip install transformers
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
#!pip install tqdm
#!pip install accelerate
#!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 2.9 MB/s eta 0:00:0000:0100:01


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import load_dataset
import pandas as pd
import gc

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC

# Import other classifiers
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Load a text classification dataset
dataset = load_dataset("imdb")

# Convert the dataset splits into pandas DataFrames
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

# Assign the relevant dataset columns to feature and label variables
X_train_text = train_df["text"]
y_train = train_df["label"]
X_test_text = test_df["text"]
y_test = test_df["label"]

# Transform the text data into numerical features using TF-IDF
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

# Convert the transformed data into a format suitable for classifiers
# (TfidfVectorizer already returns sparse matrices, which sklearn accepts directly.
#  For classifiers that need dense input we convert inside the loop.)

# Define a set of classifiers to evaluate
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, verbose=1),
    "Linear SVC": LinearSVC(max_iter=2000, verbose=1),
    "Multinomial Naive Bayes": MultinomialNB(alpha=1.0),
    "Decision Tree": DecisionTreeClassifier(max_depth=50),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, verbose=1),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=5, verbose=1),
}

# Loop through classifiers, train them, and evaluate their performance
for model_name, model in classifiers.items():
    try:
        print(f"--- Running: {model_name} ---")

        # Train the model using training data
        model.fit(X_train, y_train)

        # Generate predictions on test data
        y_pred = model.predict(X_test)

        # Generate and print a classification report
        report = classification_report(y_test, y_pred, target_names=["Negative", "Positive"])
        print(report)

        # Clean up memory after each classifier runs
        del model, y_pred
        gc.collect()

    except Exception as e:
        print(f"Error with {model_name}: {e}")
        continue  # Skip to the next classifier if an error occurs


--- Running: Logistic Regression ---
              precision    recall  f1-score   support

    Negative       0.90      0.90      0.90     12500
    Positive       0.90      0.90      0.90     12500

    accuracy                           0.90     25000
   macro avg       0.90      0.90      0.90     25000
weighted avg       0.90      0.90      0.90     25000

--- Running: Linear SVC ---
[LibLinear]...*
optimization finished, #iter = 37
Objective value = -3709.481209
nSV = 14420
              precision    recall  f1-score   support

    Negative       0.90      0.90      0.90     12500
    Positive       0.90      0.90      0.90     12500

    accuracy                           0.90     25000
   macro avg       0.90      0.90      0.90     25000
weighted avg       0.90      0.90      0.90     25000

--- Running: Multinomial Naive Bayes ---
              precision    recall  f1-score   support

    Negative       0.86      0.89      0.87     12500
    Positive       0.89      0.85     

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done 172 tasks      | elapsed:   10.3s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   11.9s finished
[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=14)]: Done 172 tasks      | elapsed:    0.5s
[Parallel(n_jobs=14)]: Done 200 out of 200 | elapsed:    0.5s finished


              precision    recall  f1-score   support

    Negative       0.85      0.87      0.86     12500
    Positive       0.86      0.85      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000

--- Running: K-Nearest Neighbors ---
              precision    recall  f1-score   support

    Negative       0.75      0.76      0.75     12500
    Positive       0.76      0.74      0.75     12500

    accuracy                           0.75     25000
   macro avg       0.75      0.75      0.75     25000
weighted avg       0.75      0.75      0.75     25000

--- Running: Gradient Boosting ---
      Iter       Train Loss   Remaining Time 
         1           1.3440            8.56m
         2           1.3091            7.56m
         3           1.2787            7.97m
         4           1.2525            7.57m
         5           1.2296            7.73m
   

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

dataset = load_dataset("imdb")


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(["text"]).rename_column("label", "labels")

tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"]

model = AutoModelForSequenceClassification.from_pretrained("bert-large-uncased", num_labels=2)

# TODO: PLAY WITH THE HYPERPARAMETERS
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.1,
    logging_dir="./logs",
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
results = trainer.evaluate(eval_dataset)
print("Evaluation Results:", results)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-large-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will

In [ ]:
# Adapt to Sentence BERT
# Strategy: use sentence-transformers/all-MiniLM-L6-v2 to produce fixed-size
# sentence embeddings, then train a simple classification head on top.

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load dataset
dataset = load_dataset("imdb")

# Load Sentence-BERT model and tokenizer
sbert_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
sbert_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model = sbert_model.to(device)
sbert_model.eval()


# Mean pooling: average token embeddings weighted by attention mask
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state  # (batch, seq_len, hidden)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1).clamp(min=1e-9)


def encode_texts(texts, tokenizer, model, batch_size=64, max_length=256):
    """Encode a list of texts into sentence embeddings using Sentence-BERT."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = model(**encoded)
        embeddings = mean_pooling(outputs, encoded["attention_mask"])
        # Normalize (as sentence-transformers does)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())
        if (i // batch_size) % 50 == 0:
            print(f"  Encoded {min(i + batch_size, len(texts))}/{len(texts)} texts")
    return np.concatenate(all_embeddings, axis=0)


# Encode train and test splits
print("Encoding training set...")
X_train_emb = encode_texts(dataset["train"]["text"], sbert_tokenizer, sbert_model)
print("Encoding test set...")
X_test_emb = encode_texts(dataset["test"]["text"], sbert_tokenizer, sbert_model)

y_train = np.array(dataset["train"]["label"])
y_test = np.array(dataset["test"]["label"])

print(f"Train embeddings shape: {X_train_emb.shape}")
print(f"Test embeddings shape:  {X_test_emb.shape}")

# Train a Logistic Regression classifier on the sentence embeddings
print("\nTraining Logistic Regression on Sentence-BERT embeddings...")
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


In [ ]:
from sklearn.metrics import classification_report


predictions = trainer.predict(eval_dataset)
logits = predictions.predictions
y_pred = np.argmax(logits, axis=1)
if "labels" in eval_dataset.column_names:
    y_true = eval_dataset["labels"]
elif "label" in eval_dataset.column_names:
    y_true = eval_dataset["label"]
else:
    raise ValueError("Neither 'labels' nor 'label' found in dataset!")




report = classification_report(y_true, y_pred, target_names=["Negative", "Positive"])  # Adjust class names as needed
print("Classification Report:")
print(report)
